# 15c: Acquire the unused EyePACS test set (53,576 images)

Supersedes 15 and 15b. Labels now come from `eyepacs/raw/retinopathy_solution.csv` in Drive:
Kaggle published the test labels in a **discussion post**
(kaggle.com/c/diabetic-retinopathy-detection/discussion/16149), not on the Data tab, so the API
cannot fetch them. Images still come from the API (`test.zip.001` to `.007`).

v1 used only the 35,126-image training portion. This adds the 53,576-image test portion, with a
published distribution of 39,533 / 3,762 / 7,861 / 1,214 / 1,206 across grades 0 to 4, so roughly
**1,206 more PDR eyes** on top of the 708 you have.

Preprocessing uses the recovered v1 pipeline (RGB, 512x512 Lanczos, PNG), verified pixel-identical
to what the models were trained on. **No model is run and nothing is evaluated.**

Cell 6 perceptually hashes every new image against all 35,126 already used, to detect reused
images. It cannot detect the same patient photographed on another date; that limit is recorded.

Prerequisites: Kaggle token as a Colab secret (`KAGGLE_API_TOKEN`, or legacy username/key); rules
accepted (done); A100 runtime; ~20 GB free in Drive and ~65 GB free local disk.

**Run Cells 1 and 2 first and stop** (Cell 2 is instant and prints the grade distribution).
Then 1.5 to 3 hours for the rest.

In [2]:
# Cell 1: mount, authenticate, disk check
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, io, json, zipfile, shutil, time, datetime, subprocess
import numpy as np, pandas as pd, PIL
from PIL import Image
!pip -q install --upgrade kaggle

def kaggle_auth():
    home=os.path.expanduser('~/.kaggle'); os.makedirs(home, exist_ok=True)
    for name in ('KAGGLE_API_TOKEN','KAGGLE_TOKEN'):
        try: tok=userdata.get(name)
        except Exception: tok=None
        if tok:
            tok=tok.strip(); os.environ['KAGGLE_API_TOKEN']=tok
            p=f'{home}/access_token'
            with open(p,'w') as f: f.write(tok)
            os.chmod(p,0o600); return f'API token via {name}'
    try: u=userdata.get('KAGGLE_USERNAME'); k=userdata.get('KAGGLE_KEY')
    except Exception: u=k=None
    if u and k:
        os.environ['KAGGLE_USERNAME']=u.strip(); os.environ['KAGGLE_KEY']=k.strip()
        p=f'{home}/kaggle.json'
        with open(p,'w') as f: json.dump({'username':u.strip(),'key':k.strip()},f)
        os.chmod(p,0o600); return 'legacy username/key'
    raise SystemExit('No Kaggle credentials found in Colab secrets.')
print('authenticated:', kaggle_auth())

BASE='/content/drive/MyDrive/stochastic_orthogonality/data'
V2=f'{BASE}/dr_ensemble_v2'
LABELS=f'{BASE}/eyepacs/raw/retinopathy_solution.csv'
OUT=f'{V2}/07_kaggle_test_images'; os.makedirs(OUT, exist_ok=True)
OLD_BATCHES=[f'{BASE}/eyepacs_full/eyepacs_processed_batch{i}.zip' for i in range(1,6)]
COMP='diabetic-retinopathy-detection'
assert os.path.exists(LABELS), f'labels not found at {LABELS}'
free=shutil.disk_usage('/content').free/1e9
print(f'free local disk {free:.0f} GB | CPU cores {os.cpu_count()} | PIL {PIL.__version__}')
assert free>65, 'need ~65 GB free local disk'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
authenticated: API token via KAGGLE_API_TOKEN
free local disk 202 GB | CPU cores 12 | PIL 11.3.0


In [3]:
# Cell 2: labels from Drive; check them before committing to 50 GB
sol=pd.read_csv(LABELS); sol.columns=[c.strip() for c in sol.columns]
print('columns:', list(sol.columns))
icol=[c for c in sol.columns if c.lower() in ('image','image_id','id')][0]
lcol=[c for c in sol.columns if c.lower() in ('level','label','grade','diagnosis')][0]
sol=sol.rename(columns={icol:'image', lcol:'level'})
sol['image']=sol['image'].astype(str)
print(sol.head(3).to_string(index=False))
print('\nrows:', len(sol))
print('grade distribution:', sol['level'].value_counts().sort_index().to_dict())
print('PDR (grade 4):', int((sol['level']==4).sum()), '| severe NPDR (grade 3):', int((sol['level']==3).sum()))
print('referable (grade>=2) rate:', round(float((sol['level']>=2).mean()),4))
if 'Usage' in sol.columns: print('Usage split:', sol['Usage'].value_counts().to_dict())
assert len(sol)>50000, f'expected ~53,576 rows, got {len(sol)}'
print('\npublished reference distribution: 39,533 / 3,762 / 7,861 / 1,214 / 1,206')

columns: ['image', 'level', 'Usage']
  image  level   Usage
 1_left      0 Private
1_right      0 Private
 2_left      0  Public

rows: 53576
grade distribution: {0: 39533, 1: 3762, 2: 7861, 3: 1214, 4: 1206}
PDR (grade 4): 1206 | severe NPDR (grade 3): 1214
referable (grade>=2) rate: 0.1919
Usage split: {'Private': 42670, 'Public': 10906}

published reference distribution: 39,533 / 3,762 / 7,861 / 1,214 / 1,206


In [4]:
# Cell 3: download the seven test parts one at a time, appending into one zip
TEST_ZIP='/content/test.zip'
if not os.path.exists(TEST_ZIP):
    with open(TEST_ZIP,'wb') as out:
        for i in range(1,8):
            part=f'test.zip.{i:03d}'; lp=f'/content/{part}'
            if not os.path.exists(lp):
                print('downloading', part, flush=True)
                subprocess.run(['kaggle','competitions','download','-c',COMP,'-f',part,'-p','/content','--force'],check=True)
                if os.path.exists(lp+'.zip'):
                    with zipfile.ZipFile(lp+'.zip') as z: z.extractall('/content')
                    os.remove(lp+'.zip')
            assert os.path.exists(lp), f'{part} did not download'
            with open(lp,'rb') as f:
                while True:
                    b=f.read(64<<20)
                    if not b: break
                    out.write(b)
            os.remove(lp)
            print(f'  appended {part}; total {os.path.getsize(TEST_ZIP)/1e9:.1f} GB; '
                  f'free {shutil.disk_usage("/content").free/1e9:.0f} GB', flush=True)
with open(TEST_ZIP,'rb') as f: print('signature:', f.read(4), '(expect PK\\x03\\x04)')
Z=zipfile.ZipFile(TEST_ZIP)
IDX={os.path.splitext(os.path.basename(n))[0]: n for n in Z.namelist() if n.lower().endswith(('.jpeg','.jpg'))}
missing=set(sol['image'])-set(IDX)
print(f'images in archive: {len(IDX)} | labelled: {len(sol)} | missing: {len(missing)}')
assert not missing, f'labelled images missing, e.g. {list(missing)[:5]}'

downloading test.zip.001
  appended test.zip.001; total 8.4 GB; free 194 GB
downloading test.zip.002
  appended test.zip.002; total 16.8 GB; free 185 GB
downloading test.zip.003
  appended test.zip.003; total 25.2 GB; free 177 GB
downloading test.zip.004
  appended test.zip.004; total 33.6 GB; free 169 GB
downloading test.zip.005
  appended test.zip.005; total 41.9 GB; free 160 GB
downloading test.zip.006
  appended test.zip.006; total 50.3 GB; free 152 GB
downloading test.zip.007
  appended test.zip.007; total 53.3 GB; free 149 GB
signature: b'PK\x03\x04' (expect PK\x03\x04)
images in archive: 53576 | labelled: 53576 | missing: 0


In [5]:
# Cell 4: v1 pipeline plus perceptual hash
from scipy.fftpack import dct
def v1_preprocess(fp):
    return Image.open(fp).convert('RGB').resize((512,512), Image.LANCZOS)
def phash(img, hs=8):
    a=np.asarray(img.convert('L').resize((32,32), Image.LANCZOS), dtype=np.float64)
    d=dct(dct(a, axis=0, norm='ortho'), axis=1, norm='ortho')[:hs,:hs]
    m=np.median(d.ravel()[1:])
    return ''.join('1' if b else '0' for b in (d>m).ravel())
probe=v1_preprocess(io.BytesIO(Z.read(IDX[sol['image'].iloc[0]])))
print('preprocessed size', probe.size, '| hash bits', len(phash(probe)))

preprocessed size (512, 512) | hash bits 64


In [6]:
# Cell 5: preprocess in four chunks; upload and free disk per chunk (resumable)
from multiprocessing import Pool
PNG='/content/kt_png'; os.makedirs(PNG, exist_ok=True)
_Z=None
def _init():
    global _Z; _Z=zipfile.ZipFile(TEST_ZIP)
def _work(args):
    iid, member = args
    try:
        img=v1_preprocess(io.BytesIO(_Z.read(member)))
        img.save(f'{PNG}/{iid}.png'); return iid, phash(img), 'ok'
    except Exception as e:
        return iid, None, f'error: {e}'

ids=sorted(sol['image']); chunks=np.array_split(ids,4); errors=[]
hp='/content/kt_hashes.json'; HASH=json.load(open(hp)) if os.path.exists(hp) else {}
for ci, chunk in enumerate(chunks,1):
    zp=f'{OUT}/eyepacs_kaggletest_batch{ci}_v2.zip'
    if os.path.exists(zp) and all(i in HASH for i in chunk):
        print(f'batch {ci} already done, skipping'); continue
    jobs=[(i, IDX[i]) for i in chunk]; t0=time.time()
    with Pool(os.cpu_count(), initializer=_init) as pool:
        for k,(iid,h,st) in enumerate(pool.imap_unordered(_work, jobs, chunksize=16),1):
            if st=='ok': HASH[iid]=h
            else: errors.append((iid,st))
            if k%3000==0: print(f'  batch {ci}: {k}/{len(jobs)} ({time.time()-t0:.0f}s)', flush=True)
    local=f'/content/eyepacs_kaggletest_batch{ci}_v2.zip'
    with zipfile.ZipFile(local,'w',zipfile.ZIP_STORED) as zw:
        for iid in chunk:
            p=f'{PNG}/{iid}.png'
            if os.path.exists(p): zw.write(p, f'{iid}.png')
    shutil.copy(local, zp); os.remove(local)
    for iid in chunk:
        p=f'{PNG}/{iid}.png'
        if os.path.exists(p): os.remove(p)
    with open(hp,'w') as f: json.dump(HASH,f)
    print(f'  batch {ci} saved ({time.time()-t0:.0f}s); free {shutil.disk_usage("/content").free/1e9:.0f} GB', flush=True)
print('processed', len(HASH), '| errors', len(errors), errors[:5])

  batch 1: 3000/13394 (128s)
  batch 1: 6000/13394 (250s)
  batch 1: 9000/13394 (371s)
  batch 1: 12000/13394 (496s)
  batch 1 saved (570s); free 146 GB
  batch 2: 3000/13394 (127s)
  batch 2: 6000/13394 (250s)
  batch 2: 9000/13394 (374s)
  batch 2: 12000/13394 (495s)
  batch 2 saved (571s); free 143 GB
  batch 3: 3000/13394 (128s)
  batch 3: 6000/13394 (252s)
  batch 3: 9000/13394 (375s)
  batch 3: 12000/13394 (498s)
  batch 3 saved (572s); free 139 GB
  batch 4: 3000/13394 (127s)
  batch 4: 6000/13394 (250s)
  batch 4: 9000/13394 (373s)
  batch 4: 12000/13394 (498s)
  batch 4 saved (575s); free 136 GB
processed 53576 | errors 0 []


In [7]:
# Cell 6: overlap check against all 35,126 images already used
def hashes_from_zip(path):
    out={}
    with zipfile.ZipFile(path) as z:
        for n in z.namelist():
            if n.endswith('.png'):
                out[os.path.splitext(os.path.basename(n))[0]]=phash(Image.open(io.BytesIO(z.read(n))))
    return out
OLD={}
for b in OLD_BATCHES:
    OLD.update(hashes_from_zip(b)); print('hashed', os.path.basename(b), '->', len(OLD), flush=True)
fresh=f'{V2}/06_fresh_images'
for fn in sorted(os.listdir(fresh)):
    if fn.endswith('.zip'):
        OLD.update(hashes_from_zip(f'{fresh}/{fn}')); print('hashed', fn, '->', len(OLD), flush=True)
by_hash={}
for k,h in OLD.items(): by_hash.setdefault(h,[]).append(k)
exact=[(k,by_hash[h]) for k,h in HASH.items() if h in by_hash]
def ham(a,b): return sum(c1!=c2 for c1,c2 in zip(a,b))
near=None
if len(exact)<200:
    old_items=list(OLD.values())[:20000]; samp=list(HASH.values())[:2000]
    near=int(sum(1 for h in samp if any(ham(h,h2)<=4 for h2 in old_items)))
OVER={'new_images_hashed':len(HASH),'previously_used_hashed':len(OLD),'exact_hash_matches':len(exact),
      'examples':[{'new':k,'old':v} for k,v in exact[:10]],
      'near_duplicates_in_2000x20000_sample':near,
      'limitation':'hashing detects reused images, not the same patient imaged on another date'}
print(json.dumps(OVER, indent=2))

hashed eyepacs_processed_batch1.zip -> 4000
hashed eyepacs_processed_batch2.zip -> 8000
hashed eyepacs_processed_batch3.zip -> 12000
hashed eyepacs_processed_batch4.zip -> 16000
hashed eyepacs_processed_batch5.zip -> 19332
hashed eyepacs_fresh_batch1_v2.zip -> 23281
hashed eyepacs_fresh_batch2_v2.zip -> 27230
hashed eyepacs_fresh_batch3_v2.zip -> 31178
hashed eyepacs_fresh_batch4_v2.zip -> 35126
{
  "new_images_hashed": 53576,
  "previously_used_hashed": 35126,
  "exact_hash_matches": 2243,
  "examples": [
    {
      "new": "10001_right",
      "old": [
        "27589_right",
        "18022_right",
        "41590_right",
        "5259_right"
      ]
    },
    {
      "new": "10037_right",
      "old": [
        "27358_right"
      ]
    },
    {
      "new": "10103_right",
      "old": [
        "41965_right",
        "7271_right",
        "9063_right"
      ]
    },
    {
      "new": "10041_right",
      "old": [
        "28143_right"
      ]
    },
    {
      "new": "10049_right"

In [8]:
# Cell 7: split file and provenance
kt=sol.copy()
kt['image_id']=kt['image']
kt['patient_id']='kt_'+kt['image'].str.split('_').str[0]
kt['eye']=kt['image'].str.split('_').str[-1]
kt['binary_label']=(kt['level']>=2).astype(int)
cols=['image_id','patient_id','eye','level','binary_label']+(['Usage'] if 'Usage' in kt.columns else [])
kt[cols].sort_values('image_id').to_csv(f'{V2}/01_splits/eyepacs_split_kaggletest_v2.csv', index=False)
REC={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'images_source':f'Kaggle {COMP}: test.zip.001-007 via API',
     'labels_source':'retinopathy_solution.csv from Kaggle discussion 16149, stored in eyepacs/raw/',
     'images':int(len(kt)),'patients_by_id':int(kt['patient_id'].nunique()),
     'grade_distribution':{int(a):int(b) for a,b in kt['level'].value_counts().sort_index().items()},
     'pdr_eyes':int((kt['level']==4).sum()),'severe_grade3_eyes':int((kt['level']==3).sum()),
     'referable_rate':round(float(kt['binary_label'].mean()),4),
     'preprocessing':'recovered v1 pipeline: RGB, 512x512 Lanczos, PNG (verified pixel-identical)',
     'overlap_check':OVER,'preprocessing_errors':len(errors),
     'patient_id_note':('Kaggle numbers train and test patients independently, so raw integer ids '
                        'collide; test patient ids are prefixed kt_. Kaggle documents train and test as '
                        'different patients; the hash check tests only for reused images.'),
     'note':'Preparation only. No model run, no evaluation.'}
with open(f'{V2}/99_audit/kaggletest_preparation_v2.json','w') as f: json.dump(REC,f,indent=2,default=str)
print(json.dumps({k:REC[k] for k in ['images','patients_by_id','grade_distribution','pdr_eyes',
                                     'severe_grade3_eyes','referable_rate']}, indent=2))
print('\nwrote 01_splits/eyepacs_split_kaggletest_v2.csv and 99_audit/kaggletest_preparation_v2.json')

{
  "images": 53576,
  "patients_by_id": 26788,
  "grade_distribution": {
    "0": 39533,
    "1": 3762,
    "2": 7861,
    "3": 1214,
    "4": 1206
  },
  "pdr_eyes": 1206,
  "severe_grade3_eyes": 1214,
  "referable_rate": 0.1919
}

wrote 01_splits/eyepacs_split_kaggletest_v2.csv and 99_audit/kaggletest_preparation_v2.json
